# AF2 spectral — Stage 1 sequential Kaggle
Menjalankan `AF2WIN → AF2ORI → AF2POL → AF2SOFT → AF2LUM` satu per satu pada satu GPU. Tidak ada test, tidak ada progress bar besar. Jika session terputus, pasang ZIP output sebelumnya sebagai Kaggle input lalu jalankan ulang; hanya arm dengan kontrak SHA yang identik akan dipulihkan.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input')
named=sorted(path for path in INPUT.rglob('D0_seed42_best.pt') if path.is_file())
historical=sorted(path for path in INPUT.rglob('best.pt') if path.parent.name=='weights' and path.parent.parent.name=='D0_seed42')
D0_INPUT=named[0] if len(named)==1 else (historical[0] if len(historical)==1 else None)
if D0_INPUT is None: raise FileNotFoundError('STOP CEPAT: Kaggle Input belum berisi D0 seed-42. Attach D0_seed42_best.pt atau D0_seed42/weights/best.pt; tidak ada clone, install, atau training yang dijalankan.')
print('D0 INPUT VALID:',D0_INPUT)


In [ ]:
import os, shutil, subprocess, sys, time
from pathlib import Path
WORK=Path('/kaggle/working'); os.chdir(WORK); REPO=WORK/'coffee-bean-detection'; INPUT=Path('/kaggle/input')
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
os.chdir(REPO)
sys.path.insert(0,str(REPO/'src'))
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
from coffee_detector.experiments.prepare_faruq_v3_kaggle import prepare_faruq_v3_kaggle_input
from coffee_detector.experiments.prepare_af2_spectral_kaggle import restore_spectral_kaggle_run
from coffee_detector.af2_spectral.audit import run_spectral_static_audit
d0_named=sorted(path for path in INPUT.rglob('D0_seed42_best.pt') if path.is_file())
d0_historical=sorted(path for path in INPUT.rglob('best.pt') if path.parent.name=='weights' and path.parent.parent.name=='D0_seed42')
D0=d0_named[0] if len(d0_named)==1 else (d0_historical[0] if len(d0_historical)==1 else None)
if D0 is None: raise FileNotFoundError('Attach D0_seed42_best.pt or D0_seed42/weights/best.pt as a Kaggle Input; AF2 is not used as a D0 substitute.')
DATA,CONTRACT=prepare_faruq_v3_kaggle_input(INPUT,WORK)
OUT=WORK/'af2-spectral-factorization-v1'; OUT.mkdir(exist_ok=True)
STATIC=OUT/'static_audit.json'
audit=run_spectral_static_audit(D0,STATIC,device='cuda:0')
assert audit['decision']=='PASS', 'STOP: static audit gagal; training tidak dijalankan.'
print('DATA:',DATA); print('STATIC:',STATIC); print('TEST:',CONTRACT['test_images_accessed'])

In [ ]:
ARMS=('AF2WIN','AF2ORI','AF2POL','AF2SOFT','AF2LUM')
def run_arm(arm):
    config=REPO/f'configs/af2_spectral/{arm}_yolo26n.yaml'
    restored=restore_spectral_kaggle_run(INPUT,OUT,arm=arm,seed=42,d0_checkpoint=D0,config=config)
    result=OUT/'val_reports'/f'{arm}_seed42_result.json'
    log=OUT/f'{arm}_seed42_run.log'
    if result.is_file():
        print(f'REUSE COMPLETE {arm}: {result}',flush=True); return
    command=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_af2_spectral_arm','--arm',arm,'--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),'--d0-checkpoint',str(D0),'--static-audit',str(STATIC),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
    print(f'START {arm} | restored={restored}',flush=True)
    with log.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
    previous_epoch=None
    while process.poll() is None:
        csv_path=OUT/arm/f'{arm}_seed42'/'results.csv'
        epoch=max(0,len(csv_path.read_text(errors='replace').splitlines())-1) if csv_path.is_file() else 0
        if epoch!=previous_epoch: print(f'{arm}: {epoch}/50 epoch | log={log}',flush=True); previous_epoch=epoch
        time.sleep(120)
    if process.returncode: 
        print('\n'.join(log.read_text(errors='replace').splitlines()[-150:])); raise RuntimeError(f'{arm} gagal: {process.returncode}')
    assert result.is_file(), f'Hasil {arm} tidak ditemukan: {result}'
    print(f'SELESAI {arm}',flush=True)
for arm in ARMS: run_arm(arm)
print('STAGE 1 COMPLETE:',[str(OUT/'val_reports'/f'{arm}_seed42_result.json') for arm in ARMS])

In [ ]:
import json
for arm in ARMS:
    result=json.loads((OUT/'val_reports'/f'{arm}_seed42_result.json').read_text())
    metrics=result['metrics']
    print(arm, {key:metrics[key] for key in ('macro_map50_95','bottom3_class_map50_95','worst_class_map50_95')})
archive=shutil.make_archive('/kaggle/working/af2-spectral-stage1-sequential-output','zip',OUT)
print('DOWNLOAD SEBELUM STOP SESSION:',archive)
print('Selanjutnya: attach ZIP ini + core dataset pada Global Decision notebook.')